# Lecture 6: From equations to compiled kernels

**The challenge.** Part I ended with a boundary value problem: $2n$
differential equations, half their boundary conditions at each end. Every
method for solving it, from Lecture 9's shooting to Lecture 15's automatic
homotopy, works the same way underneath: guess the missing initial
costates, *integrate* the canonical equations to the end, see how badly the
end conditions are missed, and guess again. A single solve needs tens to
hundreds of integrations; a random search or a continuation needs thousands.
Each must be accurate, because a sloppy integration gives a sloppy Jacobian
and Newton's method stops converging. And each must be fast.

The equations SymPy produced in Lectures 3 and 4 are neither: evaluated in
Python they cost microseconds per term, and a hand-written integrator is
either slow or inaccurate. This lecture covers the two pieces of engineering
between the mathematics and a usable solver: adaptive Runge–Kutta
integration, and turning the symbolic necessary conditions into compiled C.

**After this lecture you can**

- explain how an embedded Runge–Kutta pair controls its own error, and choose
  a method and tolerances;
- build, cache and inspect the compiled kernels of any problem, and write
  them out as plain C;
- use the forward map (`propagate`, `trajectory`, `propagate_batch`) and the
  pointwise kernels, and know what each costs.

## 1. Three generations of Runge–Kutta

In 1895 Carl Runge, a mathematician at Hannover who was better known for his
work in spectroscopy, published a way to integrate $\dot y = f(t, y)$ more
accurately than Euler's method without computing derivatives of $f$:
evaluate $f$ several times inside each step and combine the results. In 1901
Martin Wilhelm Kutta, then at Munich, worked out the general
conditions such combinations must satisfy and found the classical
fourth-order method that is still taught first. (Kutta is also the Kutta of
the Kutta–Joukowski theorem of aerodynamic lift, from the same years.)
Runge's daughter Nerina married the mathematician Richard Courant, who
reappears in Lecture 13 as the inventor of the penalty method.

A fixed-step method has a weakness: the step that is right for one part of a
trajectory is far too large or far too small for another. The cure is to
estimate the error of each step and adjust the next. Erwin Fehlberg, a German
mathematician who worked for NASA at the Marshall Space Flight Center in
Huntsville, published in 1969 the idea that made this cheap: two
Runge–Kutta formulas of different orders that share all their function
evaluations, an **embedded pair**. Their difference estimates the error for
free. In 1980 John Dormand and Peter Prince at Teesside Polytechnic found
pairs tuned so that the *higher*-order result is the one kept, and Ernst
Hairer and Gerhard Wanner built from their work the eighth-order code DOP853,
with a continuous interpolant (*dense output*) between steps. DOP853 is the
default integrator in this package.

## 2. How an adaptive step works

A step from $t_k$ to $t_k + h$ computes two estimates, $y_{k+1}$ of order
$p$ and $\hat y_{k+1}$ of order $p - 1$, from the same stages. Their
difference $e = y_{k+1} - \hat y_{k+1}$ is an estimate of the local error.
The step is accepted if the scaled error

$$
\text{err} = \Big\lVert \frac{e_i}{\text{atol} + \text{rtol}\,|y_i|} \Big\rVert_{\text{RMS}}
\le 1 ,
$$

and either way the next step is scaled by about $\text{err}^{-1/p}$. So
`rtol` sets the relative accuracy per step and `atol` the floor for components
near zero. The global error is not bounded by the tolerance, but it usually
follows it closely. A high-order method takes long steps at tight
tolerances, which is what we need: shooting wants extremals accurate to
$10^{-10}$ or better.

We will measure it shortly. First we need something fast to integrate.

In [ ]:
# Adaptive steps on an eccentric orbit: DOP853 takes short steps where the motion is fast
# (near the planet) and long ones where it is slow.
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
e = 0.8
period = 2 * np.pi                                  # a = 1, mu = 1
y0 = [1 - e, 0.0, 0.0, np.sqrt((1 + e) / (1 - e))]   # start at periapsis


def kepler(t, y):
    r3 = np.hypot(y[0], y[1]) ** 3
    return [y[2], y[3], -y[0] / r3, -y[1] / r3]


sol = solve_ivp(kepler, (0, period), y0, method="DOP853", rtol=1e-8, atol=1e-10)
fig, (left, right) = plt.subplots(1, 2, figsize=(11.5, 3.9), gridspec_kw={"width_ratios": [1.25, 1]})
dense = solve_ivp(kepler, (0, period), y0, method="DOP853", rtol=1e-10, atol=1e-12,
                  t_eval=np.linspace(0, period, 600))
left.plot(dense.y[0], dense.y[1], color=GRAY, lw=1)
left.plot(sol.y[0], sol.y[1], "o", color=BLUE, ms=4, label=f"{sol.t.size - 1} accepted steps")
left.plot(0, 0, "o", color=ORANGE, ms=10)
left.text(0.03, 0.08, "planet", fontsize=9)
left.set_aspect("equal")
left.set(xlabel="x", ylabel="y")
left.legend(frameon=False, fontsize=9, loc="upper left")
left.set_title("one orbit, e = 0.8: a dot at each step", fontsize=10, loc="left")
h = np.diff(sol.t)
r = np.hypot(sol.y[0], sol.y[1])[:-1]
right.semilogy(sol.t[:-1], h, "o-", color=BLUE, ms=3, lw=1, label="step size h")
right.semilogy(sol.t[:-1], 0.05 * r ** 1.5, color=GRAY, lw=1, ls="--", label=r"$\propto r^{3/2}$")
right.set(xlabel="t", ylabel="step size")
right.legend(frameon=False, fontsize=9)
right.set_title("step size: short near the planet, long far from it", fontsize=10, loc="left")
for a in (left, right):
    a.grid(alpha=0.25, which="both")
fig.tight_layout()
print(f"largest / smallest step: {h.max() / h.min():.0f}")

*Step-size control at work, on a Kepler orbit of eccentricity 0.8 with SciPy's DOP853, the same method the package uses. Near the planet the motion is fast and the steps are short; far from it they are long. The step size follows the orbital time scale $r^{3/2}$.*

## 3. From SymPy to C

The pipeline has three stages:

1. `derive(problem)` produces the necessary conditions as SymPy expressions
   (Lectures 2 to 4).
2. `codegen.generate_c(pmp)` prints them as C99: one function per object,
   with common subexpressions eliminated so shared terms are computed once.
3. `build.build_kernels(pmp)` compiles that C together with a Cython wrapper
   for fast-numerical-integration (FNI), the package's integrator, loads the
   result, and returns a `Kernels` object.

We use the cart-pole swing-up of Lecture 3, without the force limit.

In [ ]:
# From a problem statement to compiled kernels, and who uses them.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import itertools

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, ax = plt.subplots(figsize=(12, 3.6))
boxes = [
    (0.0, "OptimalControlProblem", "SymPy: states, controls,\ndynamics, costs, sets", "problem"),
    (2.45, "derive", "PMP: H, costates, control\nlaw, switches, arcs", "pontryagin"),
    (4.9, "generate_c", "C99, common subexpressions\neliminated, <math.h> only", "codegen"),
    (7.35, "build_kernels", "compile + Cython wrapper,\ncached by a hash", "build"),
    (9.8, "Kernels", "rhs, jacobian, control, ...\npropagate, trajectory", "build"),
]
w, hgt = 2.05, 1.25
for x, title, body, module in boxes:
    ax.add_patch(FancyBboxPatch((x, 0), w, hgt, boxstyle="round,pad=0.04", fc="white",
                                ec=BLUE, lw=1.6))
    ax.text(x + w / 2, hgt - 0.28, title, ha="center", fontsize=10.5, weight="bold",
            family="monospace")
    ax.text(x + w / 2, 0.42, body, ha="center", va="center", fontsize=8.5)
    ax.text(x + w / 2, -0.22, module, ha="center", fontsize=8, color=GRAY, family="monospace")
for (x0, *_), (x1, *_) in itertools.pairwise(boxes):
    ax.annotate("", xy=(x1 - 0.06, hgt / 2), xytext=(x0 + w + 0.06, hgt / 2),
                arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 1.2})
ax.annotate("", xy=(10.8, -1.0), xytext=(10.8, -0.35),
            arrowprops={"arrowstyle": "-|>", "color": ORANGE, "lw": 1.2})
ax.text(10.8, -1.1, "used by SingleShooting, MultiArcShooting (shooting)\nand by continuation and Homotopy (homotopy)",
        fontsize=9, color=ORANGE, ha="right", va="top", multialignment="right")
ax.text(5.9, 1.65, "once per problem: seconds (then a cache hit)", ha="center", fontsize=9,
        color=GRAY)
ax.text(10.8, 1.65, "every solve: compiled C, no Python in the loop", ha="center", fontsize=9, color=GRAY)
ax.set(xlim=(-0.2, 12.1), ylim=(-1.7, 1.9))
ax.axis("off")
fig.tight_layout()

*The pipeline. The first three stages run once per problem, and their result is cached; everything after works on the compiled kernels.*

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.build import build_kernels, cache_root
from auto_optimal_control.codegen import generate_c
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import OptimalControlProblem

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
q1, q2, w1, w2, u = sp.symbols("q_1 q_2 qd_1 qd_2 u", real=True)
m1, m2, l, g, d, T = sp.symbols("m_1 m_2 l g d T", positive=True)
den = m1 + m2 * (1 - sp.cos(q2) ** 2)
q1dd = (l * m2 * sp.sin(q2) * w2**2 + u + m2 * g * sp.cos(q2) * sp.sin(q2)) / den
q2dd = -(l * m2 * sp.cos(q2) * sp.sin(q2) * w2**2 + u * sp.cos(q2)
         + (m1 + m2) * g * sp.sin(q2)) / (l * den)
cart_pole = OptimalControlProblem(
    name="cart_pole_effort", time=t, states=[q1, q2, w1, w2], controls=[u],
    params=[m1, m2, l, g, d, T], dynamics=[w1, w2, q1dd, q2dd], running_cost=u**2,
    initial_state=[0, 0, 0, 0], final_state={q1: d, q2: sp.pi, w1: 0, w2: 0}, final_time=T,
)
pmp = derive(cart_pole)
prefix, header, source, macros = generate_c(pmp)
names = [line.split("(")[0].split()[-1].replace(prefix, "")
         for line in header.splitlines() if line.startswith(("int ", "double "))]
print(f"{len(source.splitlines())} lines of C defining {len(names)} functions:")
print(", ".join(names))

Each of those is a plain C function of `(t, z, p)` or of the boundary values.
The `boundary_*` and `singular_*` ones are for constrained and singular arcs
(Lectures 12 and 14); for this problem they are stubs.
Here is the right-hand side $F(t, z, p)$ of the canonical equations. The
`x0`, `x1`, ... are the common subexpressions:

In [ ]:
start = source.index(f"int {prefix}rhs(")
print(source[start : source.index("\n}", start) + 2][:1800], "...")

`build_kernels` compiles it. The build is cached by a hash of the generated
code and the toolchain: the first build takes seconds, every later one
(in any process) loads instantly. The cache lives in
`~/.cache/auto-optimal-control`, or wherever `$AUTO_OPTIMAL_CONTROL_CACHE`
points; `cache_dir=` overrides it per call, `force=True` rebuilds and
`verbose=True` shows the compiler output.

In [ ]:
start = time.perf_counter()
k = build_kernels(pmp)
print(f"build (or cache load): {time.perf_counter() - start:.2f} s")
print("cache root:", cache_root())
print(k)
print("states:", k.N_STATES, " z:", k.N_Z, " params:", k.param_names)

The C stands on its own: it includes only `<math.h>`. `write_c` saves the
header and source for use outside Python, for instance on an embedded
target.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    k.write_c(tmp)
    for f in sorted(Path(tmp).iterdir()):
        print(f.name, f.stat().st_size, "bytes")

## 4. The pointwise kernels and the forward map

The pointwise kernels evaluate the necessary conditions at one point: `rhs`
($F$), `jacobian` ($\partial F / \partial z$), `control` ($u^*$),
`hamiltonian`, `cost_rate` ($L$ at $u^*$), and more that later lectures use.
They agree with a pure-Python reference, `numeric.NumericSystem`, which
lambdifies the same expressions with SymPy (the package's tests compare the
two everywhere):

In [ ]:
from auto_optimal_control.numeric import NumericSystem

p = [1.0, 0.3, 0.5, 9.81, 1.0, 2.0]                 # m1, m2, l, g, d, T
lam0 = np.array([3.6, -52.0, 4.85, 4.29])           # a costate guess
z0 = k.z0(lam0, p)                                  # [x0, lam0] with the fixed x0
num = NumericSystem(pmp)
print("z0 =", z0)
print("|rhs C - rhs NumPy|      =", np.abs(k.rhs(0.0, z0, p) - num.rhs(0.0, z0, p)).max())
print("u*, H at z0:", k.control(0.0, z0, p), k.hamiltonian(0.0, z0, p))

The **forward map** is the core operation of the package: from an initial
state and costate, integrate the canonical equations and report what
happened. `propagate` returns the final point and much else (later lectures
use the rest); `trajectory` samples the extremal densely, with the controls
and $H$, for plots and diagnostics.

In [ ]:
r = k.propagate(0.0, z0, 2.0, p, cost=True)
print("keys:", sorted(r))
print("z(T) =", np.round(r["z"], 4), " cost =", round(r["cost"], 4), " status:", r["status"],
      " RHS evaluations:", r["nfev"])

tr = k.trajectory(0.0, z0, 2.0, p, n_points=201)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].plot(tr["t"], tr["z"][:, 0], color=BLUE, lw=2)
axes[1].plot(tr["t"], tr["z"][:, 1], color=BLUE, lw=2)
axes[2].plot(tr["t"], tr["H"] - tr["H"][0], color=ORANGE, lw=2)
for ax, title in zip(axes, ["cart $q_1$", "pole angle $q_2$", "$H(t) - H(0)$"]):
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel("t")
    ax.grid(alpha=0.3)
fig.tight_layout()

This costate guess happens to be close to an extremal: the pole swings up
and the cart arrives near $q_1 = 1$. $H$ is constant to about $10^{-8}$ of
its value, as Lecture 3 predicted for an autonomous problem; that is a free
check of the integration's accuracy. (The spikes are the dense-output
interpolation between steps; at the steps themselves $H$ is constant to the
tolerance.)

### Accuracy against work

Now the integrators. `propagate` takes `method` (`"RK23"`, `"RK45"` or
`"DOP853"`) and `rtol`, `atol`. Measure the error at $t = T$ against a
reference computed at a much tighter tolerance, and the number of right-hand
side evaluations each run costs:

In [ ]:
ref = k.propagate(0.0, z0, 2.0, p, rtol=1e-14, atol=1e-14)["z"]
fig, ax = plt.subplots(figsize=(6, 3.6))
for method, color in [("RK23", GRAY), ("RK45", ORANGE), ("DOP853", BLUE)]:
    errs, work = [], []
    for rtol in np.logspace(-3, -12, 10):
        rr = k.propagate(0.0, z0, 2.0, p, method=method, rtol=rtol, atol=rtol * 1e-2)
        errs.append(np.abs(rr["z"] - ref).max())
        work.append(rr["nfev"])
    ax.loglog(work, errs, "o-", color=color, ms=4, label=method)
ax.set(xlabel="right-hand side evaluations", ylabel="error at t = T")
ax.legend(frameon=False)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()

At loose tolerances the methods are similar; at the tolerances shooting
needs, the eighth-order DOP853 is cheaper by one or two orders of magnitude.
That is why it is the default, with `rtol=1e-10, atol=1e-12` for `propagate`
and `1e-12` in the shooting layers.

### Speed

Finally, what compilation buys. The same extremal, integrated three ways:
with the NumPy reference and FNI's `solve_ivp` (every evaluation calls back
into Python); with `propagate` (all in C, the GIL released); and with
`propagate_batch`, which integrates many initial costates at once, across
threads, without the GIL. Each row of a batch is bit-identical to the
single propagation.

In [ ]:
from fast_numerical_integration.integrators import solve_ivp

start = time.perf_counter()
ref_py = solve_ivp(num.ode(p), (0.0, 2.0), z0, method="DOP853", rtol=1e-10, atol=1e-12)
t_py = time.perf_counter() - start

start = time.perf_counter()
for _ in range(100):
    k.propagate(0.0, z0, 2.0, p)
t_c = (time.perf_counter() - start) / 100

rng = np.random.default_rng(0)
Z0 = np.array([k.z0(lam0 + rng.normal(0, 0.5, 4), p) for _ in range(2000)])
timings = {}
for n_threads in (1, 4):
    start = time.perf_counter()
    batch = k.propagate_batch(0.0, Z0, 2.0, p, n_threads=n_threads)
    timings[n_threads] = (time.perf_counter() - start) / len(Z0)
same = np.array_equal(batch["z"][0], k.propagate(0.0, Z0[0], 2.0, p)["z"])
print(f"NumPy reference + solve_ivp: {1e3 * t_py:8.3f} ms per extremal")
print(f"k.propagate:                 {1e3 * t_c:8.3f} ms per extremal")
for n, tt in timings.items():
    print(f"k.propagate_batch, {n} thread(s): {1e3 * tt:6.3f} ms per extremal")
print("batch row identical to propagate:", same)

Two to three orders of magnitude separate Python from C. That gap is what
makes random searches over thousands of costates (Lecture 9) or long
continuations (Lecture 10) affordable, and what a learned sampler of
costates would rely on.

### FNI objects

The kernels also expose native FNI objects, for anyone who wants to drive the
integrator directly: `k.Dynamics(params)` is an `OdeSystem` for FNI's
`solve_ivp` and `integrate`, and its `params` can be changed between solves
without recompiling.

In [ ]:
dyn = k.Dynamics(p)
sol = solve_ivp(dyn, (0.0, 2.0), z0, method="DOP853", rtol=1e-10, atol=1e-12)
print("native solve_ivp vs propagate, at the tolerance:", np.abs(sol.y[:, -1] - r["z"]).max())
dyn.params = [1.0, 0.3, 0.6, 9.81, 1.0, 2.0]       # a longer pole, same compiled code
sol2 = solve_ivp(dyn, (0.0, 2.0), z0, method="DOP853", rtol=1e-10, atol=1e-12)
print("final pole angle with l = 0.5 and l = 0.6:", sol.y[1, -1], sol2.y[1, -1])

## Summary

- Embedded Runge–Kutta pairs estimate their own error and adapt the step;
  `rtol` and `atol` set the per-step accuracy. DOP853 (eighth order) is the
  default, because shooting needs tight tolerances.
- `generate_c` prints the necessary conditions as C with common
  subexpressions eliminated; `build_kernels` compiles, caches and loads them;
  `write_c` exports plain C.
- `propagate` is the forward map (final point, cost, status, evaluation
  count, and more); `trajectory` samples it; `propagate_batch` runs many in
  parallel, bit-identical to single runs; `NumericSystem` is the
  pure-Python reference.
- Compiled kernels are hundreds of times faster than lambdified Python, which
  is what makes the rest of the course practical.

## Exercises

1. Plot $H(t) - H(0)$ along the extremal for `rtol` from $10^{-6}$ to
   $10^{-12}$. How does the drift scale with the tolerance?
2. Time `propagate_batch` with 1, 2, 4 and 8 threads on your machine. Where
   does the speed-up stop, and why?
3. Integrate the same extremal *backwards*, from $z(T)$ to $t = 0$. How close
   do you get to $z_0$? Repeat with a costate guess far from an extremal.
   What does this say about the sensitivity Lecture 8 measures?

## Further reading

- E. Hairer, S. P. Nørsett and G. Wanner, *Solving Ordinary Differential
  Equations I* (Springer, 1993), chapter II: Runge–Kutta methods, embedded
  pairs, DOP853 and dense output.
- J. C. Butcher, "A history of Runge–Kutta methods", *Applied Numerical
  Mathematics* 20, 1996.
- The fast-numerical-integration README, for the integrator's C API.